# Spatial Leakage Experiment

This notebook demonstrates the impact of **Spatial Leakage** in satellite imagery datasets.

## What is Spatial Leakage?
Satellite image datasets are often created by dividing large geographic areas into smaller image tiles. Tiles that are physically adjacent share highly correlated features, identical terrain patterns, lighting conditions, and atmospheric effects.

If we perform a simple random split to create our training and validation sets, it is highly likely that adjacent tiles will end up in opposite sets. This leads to information "leaking" from the training set into the validation set. As a result, the validation metrics become artificially inflated because the model is essentially being tested on data it has already seen (or data that is functionally identical to what it has seen).

## The Experiment
To quantify this effect, we compare two splitting strategies:

1. **Random Split**: Shuffling all 27,000 images and randomly assigning 20% to the validation set.
2. **Block Split**: Grouping sequential images (which act as a proxy for spatial adjacency) into blocks, and splitting the dataset at the block level rather than the individual tile level. We did this in `data_setup.py`.

We expect the **Random Split** to yield a higher validation accuracy due to spatial leakage, whereas the **Block Split** provides a more realistic estimate of the model's generalization capabilities to unseen geographic regions.

In [1]:
import os
import sys
sys.path.append('..')
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from models import BaselineCNN
import torch.optim as optim
from train_classifier import train_epoch, evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = '../data'
print(f'Using device: {DEVICE}')

Using device: cpu


In [2]:
# Load full dataset
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, '2750'), transform=transform)
print(f'Total images: {len(full_dataset)}')

Total images: 27000


In [3]:
# 1. Random Split
train_idx_rand, val_idx_rand = train_test_split(list(range(len(full_dataset))), test_size=0.2, random_state=42)
train_loader_rand = DataLoader(Subset(full_dataset, train_idx_rand), batch_size=64, shuffle=True)
val_loader_rand = DataLoader(Subset(full_dataset, val_idx_rand), batch_size=64, shuffle=False)

# 2. Block Split (Using our splits from data_setup.py)
from train_classifier import get_dataloaders
import train_classifier
train_classifier.DATA_DIR = '../data'
train_loader_block, val_loader_block, _ = train_classifier.get_dataloaders()
print('Data loaders ready.')

Data loaders ready.


In [4]:
def quick_train_and_evaluate(train_loader, val_loader, name):
    print(f"\n--- Training with {name} ---")
    model = BaselineCNN(num_classes=10).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Train for 2 epochs (sufficient to demonstrate the leakage effect)
    for epoch in range(2):
        loss = train_epoch(model, train_loader, criterion, optimizer)
        print(f'Epoch {epoch+1}/2 - Train Loss: {loss:.4f}', flush=True)
    
    _, val_f1, _, _ = evaluate(model, val_loader, criterion)
    print(f"{name} - Final Validation Macro F1: {val_f1:.4f}")
    return val_f1

f1_rand = quick_train_and_evaluate(train_loader_rand, val_loader_rand, "Random Split")
f1_block = quick_train_and_evaluate(train_loader_block, val_loader_block, "Block Split")

print(f"\n========================================")
print(f"Random Split F1:  {f1_rand:.4f}")
print(f"Block Split F1:   {f1_block:.4f}")
print(f"Difference (Inflation due to leakage): {f1_rand - f1_block:.4f}")
print(f"========================================")
print(f"\nConclusion: Random split inflates validation F1 by ~{(f1_rand - f1_block)*100:.1f}% due to spatial leakage.")


--- Training with Random Split ---


  Batch 50/338 - Loss: 1.7304


  Batch 100/338 - Loss: 1.3055


  Batch 150/338 - Loss: 1.2219


  Batch 200/338 - Loss: 1.3788


...

  Batch 200/338 - Loss: 0.8506


  Batch 250/338 - Loss: 0.7265


  Batch 300/338 - Loss: 0.6547


Epoch 2/2 - Train Loss: 0.7836


Block Split - Final Validation Macro F1: 0.7483

Random Split F1:  0.7092
Block Split F1:   0.7483
Difference (Inflation due to leakage): -0.0391

Conclusion: Random split inflates validation F1 by ~-3.9% due to spatial leakage.
